<a href="https://colab.research.google.com/github/RatchanonPa/Data-Warehouse-and-Big-Data-Analytics/blob/main/Income_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Import Libraries

In [28]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [5]:
pd.set_option("display.max_columns", None)

# Load data


In [2]:
!unzip /content/data-science-income-prediction.zip

Archive:  /content/data-science-income-prediction.zip
  inflating: column_desctiptions.csv  
  inflating: sample_submission.csv   
  inflating: test.csv                
  inflating: train.csv               


In [32]:
train = pd.read_csv('/content/train.csv')
test = pd.read_csv('/content/test.csv')
describtion = pd.read_csv('/content/column_desctiptions.csv')

<ipython-input-32-7085e7acec37>:1: DtypeWarning: Columns (19) have mixed types. Specify dtype option on import or set low_memory=False.
  train = pd.read_csv('/content/train.csv')


In [7]:
train

,id,address_role,zip_code,deposit_balance,deposit_incoming,deposit_outgoing,average_loan_24_months,average_outstanding_balance_6_months,children,credit_history_3_years,credit_history_1_year,date_of_birth,education,employment_time,employment_industry,family_status,gender,housing_type,income_type,marital_status,max_outstanding_balance_12_months,total_debt,primary_income
0,0,NaN,Unspecified,NaN,NaN,NaN,NaN,780.00,NaN,NaN,NaN,1955-11-01,Unspecified,NaN,NaN,NaN,F,NaN,Retired Pensioner,NaN,9200.0,0.0,30000.0
1,1,NaN,Unspecified,NaN,NaN,NaN,NaN,36017.00,NaN,NaN,NaN,1975-03-01,Unspecified,NaN,NaN,NaN,F,NaN,Salaried Govt,NaN,104754.4,19067.0,59000.0
2,2,NaN,Unspecified,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1948-01-01,Unspecified,NaN,NaN,NaN,M,NaN,Retired Pensioner,NaN,NaN,18781.0,30000.0
3,3,NaN,Unspecified,NaN,NaN,NaN,NaN,8100.00,NaN,NaN,NaN,1974-08-01,Higher education,More than five years,Other,Married,F,NaN,Employed,NaN,51800.0,0.0,60000.0
4,4,NaN,Unspecified,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1985-03-01,Secondary / secondary special,More than five years,Other,Married,F,NaN,Salaried Govt,NaN,NaN,0.0,40000.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1299995,1299995,NaN,Unspecified,NaN,NaN,NaN,NaN,65571.31,NaN,NaN,NaN,1990-10-01,Secondary / secondary special,Less than one year,Education,Married,F,NaN,Salaried Govt,NaN,75040.6,56061.6,70000.0
1299996,1299996,NaN,Unspecified,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1962-05-01,Higher education,More than five years,Other,Married,M,NaN,Employed,NaN,NaN,0.0,28000.0
1299997,1299997,NaN,Unspecified,NaN,NaN,NaN,NaN,8530.36,NaN,2.0,2.0,1995-01-01,Secondary / secondary special,More than one year,Education,Single,M,NaN,Salaried Govt,NaN,19961.8,15134.4,50000.0
1299998,1299998,NaN,Unspecified,NaN,NaN,NaN,NaN,239373.66,NaN,NaN,NaN,1960-10-01,Unspecified,NaN,NaN,NaN,F,NaN,Retired Pensioner,NaN,312925.8,221552.8,90000.0


In [8]:
print(train.shape)
print(train.columns)
train.duplicated().sum()

(1300000, 23)
Index(['id', 'address_role', 'zip_code', 'deposit_balance', 'deposit_incoming',
       'deposit_outgoing', 'average_loan_24_months',
       'average_outstanding_balance_6_months', 'children',
       'credit_history_3_years', 'credit_history_1_year', 'date_of_birth',
       'education', 'employment_time', 'employment_industry', 'family_status',
       'gender', 'housing_type', 'income_type', 'marital_status',
       'max_outstanding_balance_12_months', 'total_debt', 'primary_income'],
      dtype='object')


0

In [9]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1300000 entries, 0 to 1299999
Data columns (total 23 columns):
 #   Column                                Non-Null Count    Dtype  
---  ------                                --------------    -----  
 0   id                                    1300000 non-null  int64  
 1   address_role                          10462 non-null    object 
 2   zip_code                              1221957 non-null  object 
 3   deposit_balance                       43512 non-null    float64
 4   deposit_incoming                      43512 non-null    float64
 5   deposit_outgoing                      43512 non-null    float64
 6   average_loan_24_months                138241 non-null   float64
 7   average_outstanding_balance_6_months  583462 non-null   float64
 8   children                              0 non-null        float64
 9   credit_history_3_years                31145 non-null    float64
 10  credit_history_1_year                 31145 non-null  

In [10]:
train.describe()

,id,deposit_balance,deposit_incoming,deposit_outgoing,average_loan_24_months,average_outstanding_balance_6_months,children,credit_history_3_years,credit_history_1_year,max_outstanding_balance_12_months,total_debt,primary_income
count,1.300000e+06,4.351200e+04,4.351200e+04,4.351200e+04,138241.000000,5.834620e+05,0.0,31145.000000,31145.000000,6.466800e+05,1.299998e+06,1.300000e+06
mean,6.499995e+05,1.013948e+04,3.080157e+03,3.743755e+03,44800.212186,4.600200e+04,NaN,4.374249,2.418751,7.138220e+04,1.969506e+04,5.771512e+04
std,3.752778e+05,8.899292e+04,4.365600e+04,5.105979e+04,44886.185880,6.406960e+04,NaN,5.807446,3.547161,8.192607e+04,5.082154e+04,3.335821e+04
min,0.000000e+00,-3.357180e+05,0.000000e+00,0.000000e+00,0.000000,-7.588198e+06,NaN,0.000000,0.000000,-7.588198e+06,0.000000e+00,0.000000e+00
25%,3.249998e+05,0.000000e+00,0.000000e+00,0.000000e+00,15719.200000,8.712629e+03,NaN,0.000000,0.000000,2.006915e+04,0.000000e+00,3.600000e+04
50%,6.499995e+05,0.000000e+00,0.000000e+00,1.800000e+00,28477.800000,2.279600e+04,NaN,2.000000,1.000000,4.256700e+04,0.000000e+00,5.000000e+04
75%,9.749992e+05,2.900500e+02,0.000000e+00,5.400000e+00,56548.000000,5.552337e+04,NaN,6.000000,3.000000,9.010341e+04,1.354215e+04,7.000000e+04
max,1.299999e+06,4.219594e+06,4.180150e+06,4.622918e+06,513520.000000,1.131136e+06,NaN,57.000000,41.000000,1.235313e+06,1.210629e+06,2.000000e+05


In [11]:
test

,id,address_role,zip_code,deposit_balance,deposit_incoming,deposit_outgoing,average_loan_24_months,average_outstanding_balance_6_months,children,credit_history_3_years,credit_history_1_year,date_of_birth,education,employment_time,employment_industry,family_status,gender,housing_type,income_type,marital_status,max_outstanding_balance_12_months,total_debt
0,1300000,NaN,Unspecified,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1954-10-01,Unspecified,NaN,NaN,NaN,F,NaN,Retired Pensioner,NaN,NaN,144547.800
1,1300001,NaN,NaN,NaN,NaN,NaN,NaN,123391.960,NaN,NaN,NaN,1990-07-01,Higher education,More than one year,Other,Single,F,Owned,Private Sector Employee,NaN,138828.14,115545.740
2,1300002,NaN,Unspecified,NaN,NaN,NaN,NaN,20909.459,NaN,NaN,NaN,1991-08-01,Unspecified,NaN,NaN,NaN,M,NaN,Employed,NaN,30966.60,15446.601
3,1300003,NaN,Unspecified,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1991-11-01,Secondary / secondary special,More than five years,Other,Single,M,NaN,Employed,NaN,NaN,0.000
4,1300004,NaN,Unspecified,NaN,NaN,NaN,NaN,196100.380,NaN,NaN,NaN,1963-12-01,Unspecified,NaN,NaN,NaN,F,NaN,Employed,NaN,328052.78,166707.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
226654,1526654,NaN,Unspecified,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1985-11-01,Unspecified,NaN,NaN,NaN,F,NaN,Private Sector Employee,NaN,NaN,0.000
226655,1526655,NaN,Unspecified,NaN,NaN,NaN,NaN,13100.000,NaN,NaN,NaN,1953-06-01,Secondary / secondary special,NaN,NaN,Widowed,F,NaN,Retired Pensioner,NaN,64720.00,0.000
226656,1526656,NaN,Unspecified,NaN,NaN,NaN,NaN,78018.700,NaN,NaN,NaN,1992-08-01,Unspecified,NaN,NaN,NaN,M,NaN,Private Sector Employee,NaN,92793.28,65223.200
226657,1526657,NaN,Unspecified,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1968-03-01,Higher education,More than one year,Other,Married,F,NaN,Salaried Govt,NaN,NaN,0.000


In [12]:
print(test.shape)
print(test.columns)
test.duplicated().sum()

(226659, 22)
Index(['id', 'address_role', 'zip_code', 'deposit_balance', 'deposit_incoming',
       'deposit_outgoing', 'average_loan_24_months',
       'average_outstanding_balance_6_months', 'children',
       'credit_history_3_years', 'credit_history_1_year', 'date_of_birth',
       'education', 'employment_time', 'employment_industry', 'family_status',
       'gender', 'housing_type', 'income_type', 'marital_status',
       'max_outstanding_balance_12_months', 'total_debt'],
      dtype='object')


0

In [13]:
test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 226659 entries, 0 to 226658
Data columns (total 22 columns):
 #   Column                                Non-Null Count   Dtype  
---  ------                                --------------   -----  
 0   id                                    226659 non-null  int64  
 1   address_role                          1801 non-null    object 
 2   zip_code                              213084 non-null  object 
 3   deposit_balance                       7597 non-null    float64
 4   deposit_incoming                      7597 non-null    float64
 5   deposit_outgoing                      7597 non-null    float64
 6   average_loan_24_months                24268 non-null   float64
 7   average_outstanding_balance_6_months  102016 non-null  float64
 8   children                              0 non-null       float64
 9   credit_history_3_years                5369 non-null    float64
 10  credit_history_1_year                 5369 non-null    float64
 11  

# Preprocessing

In [14]:
train.isnull().sum()

,0
id,0
address_role,1289538
zip_code,78043
deposit_balance,1256488
deposit_incoming,1256488
deposit_outgoing,1256488
average_loan_24_months,1161759
average_outstanding_balance_6_months,716538
children,1300000
credit_history_3_years,1268855


In [15]:
# prompt: null คิดเป็นกี่ % ของข้อมูลในแต่ละ column

# Calculate the percentage of null values in each column
train_null_percentage = train.isnull().sum() * 100 / len(train)

# Print the results
train_null_percentage


,0
id,0.000000
address_role,99.195231
zip_code,6.003308
deposit_balance,96.652923
deposit_incoming,96.652923
deposit_outgoing,96.652923
average_loan_24_months,89.366077
average_outstanding_balance_6_months,55.118308
children,100.000000
credit_history_3_years,97.604231


In [16]:
# Calculate the percentage of null values in each column
test_null_percentage = test.isnull().sum() * 100 / len(train)

# Print the results
test_null_percentage

,0
id,0.000000
address_role,17.296769
zip_code,1.044231
deposit_balance,16.850923
deposit_incoming,16.850923
deposit_outgoing,16.850923
average_loan_24_months,15.568538
average_outstanding_balance_6_months,9.587923
children,17.435308
credit_history_3_years,17.022308


In [33]:
# 🗑️ ลบคอลัมน์ที่ Missing มากกว่า 90%
drop_cols = ['address_role', 'deposit_balance', 'deposit_incoming', 'deposit_outgoing',
             'credit_history_3_years', 'credit_history_1_year', 'housing_type', 'marital_status', 'children', 'zip_code']
train.drop(columns=drop_cols, inplace=True)
test.drop(columns=drop_cols, inplace=True)

In [37]:
# prompt: groupby train['employment_time']

train.groupby('employment_time').count()


,id,average_loan_24_months,average_outstanding_balance_6_months,date_of_birth,education,employment_industry,family_status,gender,income_type,max_outstanding_balance_12_months,total_debt,primary_income
employment_time,,,,,,,,,,,,
Less than one year,26023,1444,3760,26023,26023,25913,26023,26023,26023,4005,26023,26023
More than five years,316178,24598,84844,316178,316178,311768,316178,316178,316178,91842,316177,316178
More than one year,107798,7735,22564,107798,107798,107034,107798,107798,107798,24051,107797,107798


In [36]:
train

,id,average_loan_24_months,average_outstanding_balance_6_months,date_of_birth,education,employment_time,employment_industry,family_status,gender,income_type,max_outstanding_balance_12_months,total_debt,primary_income
0,0,NaN,780.00,1955-11-01,Unspecified,NaN,NaN,NaN,F,Retired Pensioner,9200.0,0.0,30000.0
1,1,NaN,36017.00,1975-03-01,Unspecified,NaN,NaN,NaN,F,Salaried Govt,104754.4,19067.0,59000.0
2,2,NaN,NaN,1948-01-01,Unspecified,NaN,NaN,NaN,M,Retired Pensioner,NaN,18781.0,30000.0
3,3,NaN,8100.00,1974-08-01,Higher education,More than five years,Other,Married,F,Employed,51800.0,0.0,60000.0
4,4,NaN,NaN,1985-03-01,Secondary / secondary special,More than five years,Other,Married,F,Salaried Govt,NaN,0.0,40000.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1299995,1299995,NaN,65571.31,1990-10-01,Secondary / secondary special,Less than one year,Education,Married,F,Salaried Govt,75040.6,56061.6,70000.0
1299996,1299996,NaN,NaN,1962-05-01,Higher education,More than five years,Other,Married,M,Employed,NaN,0.0,28000.0
1299997,1299997,NaN,8530.36,1995-01-01,Secondary / secondary special,More than one year,Education,Single,M,Salaried Govt,19961.8,15134.4,50000.0
1299998,1299998,NaN,239373.66,1960-10-01,Unspecified,NaN,NaN,NaN,F,Retired Pensioner,312925.8,221552.8,90000.0
